In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import kagglehub

In [7]:
path = kagglehub.dataset_download("abyssmoron/bnm5678")
print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/abyssmoron/bnm5678


In [14]:
df = pd.read_csv(os.path.join(path, "led", "Ledger.csv"), low_memory=False)
df.head()

,SlNo.,ClientID,CreatedAt,MemberID,BillID,LedgerType,Narration,Debit,Credit,Balance,PreviousBalance,ReferenceID
0,13096396.0,P71057,27 Aug 2026 23:59:58,151,270826_47660835,BILL,Trade bill for BTCUSD,123,0.00,10244.65,10367.85,NaN
1,13096395.0,M155101,27 Aug 2026 23:59:55,151,4cad1066-a245-11f1-9131-2a83bccb88bd,WITHDRAW,Ledger entry for WITHDRAW,5000,0.00,4693.52,9693.52,NaN
2,13096394.0,P62149,27 Aug 2026 23:59:52,151,270826_47660834,BILL,Trade bill for BTCUSD,0,353.41,12655.67,12302.26,NaN
3,13096392.0,M129746,27 Aug 2026 23:59:47,151,270826_47660832,BILL,Trade bill for XRPUSD,158,0.00,11380.35,11537.85,NaN
4,13096391.0,M129746,27 Aug 2026 23:59:47,151,270826_47660831,BILL,Trade bill for BTCUSD,62,0.00,11537.85,11600.25,NaN


In [15]:
df.columns

Index(['SlNo.', 'ClientID', 'CreatedAt', 'MemberID', 'BillID', 'LedgerType',
       'Narration', 'Debit', 'Credit', 'Balance', 'PreviousBalance',
       'ReferenceID'],
      dtype='object')

In [16]:
TRADE_TYPES = ["BILL"]

In [17]:
led = df[["ClientID", "CreatedAt", "LedgerType"]]
led = led[led["LedgerType"].isin(TRADE_TYPES)].copy()
led.head()

,ClientID,CreatedAt,LedgerType
0,P71057,27 Aug 2026 23:59:58,BILL
2,P62149,27 Aug 2026 23:59:52,BILL
3,M129746,27 Aug 2026 23:59:47,BILL
4,M129746,27 Aug 2026 23:59:47,BILL
5,S39238,27 Aug 2026 23:59:47,BILL


In [18]:
led["CreatedAt"] = pd.to_datetime(led["CreatedAt"], format="%d %b %Y %H:%M:%S")
led["Month"] = led["CreatedAt"].dt.to_period("M")
led.head()

,ClientID,CreatedAt,LedgerType,Month
0,P71057,2026-08-27 23:59:58,BILL,2026-08
2,P62149,2026-08-27 23:59:52,BILL,2026-08
3,M129746,2026-08-27 23:59:47,BILL,2026-08
4,M129746,2026-08-27 23:59:47,BILL,2026-08
5,S39238,2026-08-27 23:59:47,BILL,2026-08


In [19]:
activity = led[["ClientID", "Month"]].drop_duplicates()

In [20]:
activity["Cohort"] = activity.groupby("ClientID")["Month"].transform("min")

In [21]:
activity["MonthIdx"] = (activity["Month"] - activity["Cohort"]).apply(lambda x: x.n)

In [22]:
counts = (activity.groupby(["Cohort", "MonthIdx"])["ClientID"]
          .nunique()
          .unstack(fill_value=0))

In [23]:
# 7. Cohort size = clients whose first bill was in that month (= M0)
size = counts[0]

# 8. Retention %
retention = counts.div(size, axis=0) * 100

# 9. Blank out months that haven't happened yet (like the "—" cells)
last_month = activity["Month"].max()
for cohort in retention.index:
    max_idx = (last_month - cohort).n
    retention.loc[cohort, retention.columns > max_idx] = np.nan

# 10. Assemble the final table
retention.columns = [f"M{c}" for c in retention.columns]
cohort_table = retention.round(0)
cohort_table.insert(0, "Size", size)
cohort_table.index = cohort_table.index.strftime("%b %Y")
cohort_table.index.name = "Cohort"

cohort_table

,Size,M0,M1,M2,M3,M4,M5,M6,M7,M8,M9,M10,M11
Cohort,,,,,,,,,,,,,
Sep 2025,719,100.0,68.0,44.0,40.0,38.0,33.0,30.0,28.0,29.0,30.0,29.0,27.0
Oct 2025,3125,100.0,55.0,44.0,41.0,37.0,34.0,30.0,29.0,29.0,28.0,26.0,NaN
Nov 2025,2624,100.0,54.0,43.0,36.0,33.0,28.0,27.0,28.0,25.0,23.0,NaN,NaN
Dec 2025,2529,100.0,60.0,43.0,37.0,32.0,28.0,29.0,28.0,24.0,NaN,NaN,NaN
Jan 2026,2870,100.0,60.0,43.0,34.0,30.0,30.0,28.0,26.0,NaN,NaN,NaN,NaN
Feb 2026,2723,100.0,59.0,42.0,38.0,36.0,32.0,29.0,NaN,NaN,NaN,NaN,NaN
Mar 2026,2483,100.0,54.0,39.0,34.0,32.0,27.0,NaN,NaN,NaN,NaN,NaN,NaN
Apr 2026,2738,100.0,55.0,41.0,35.0,31.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
May 2026,2026,100.0,58.0,40.0,33.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
(cohort_table.style
    .format({c: "{:.0f}%" for c in cohort_table.columns if c != "Size"}, na_rep="—")
    .background_gradient(cmap="Blues", subset=cohort_table.columns[1:], vmin=0, vmax=100))

,Size,M0,M1,M2,M3,M4,M5,M6,M7,M8,M9,M10,M11
Cohort,,,,,,,,,,,,,
Sep 2025,719,100%,68%,44%,40%,38%,33%,30%,28%,29%,30%,29%,27%
Oct 2025,3125,100%,55%,44%,41%,37%,34%,30%,29%,29%,28%,26%,—
Nov 2025,2624,100%,54%,43%,36%,33%,28%,27%,28%,25%,23%,—,—
Dec 2025,2529,100%,60%,43%,37%,32%,28%,29%,28%,24%,—,—,—
Jan 2026,2870,100%,60%,43%,34%,30%,30%,28%,26%,—,—,—,—
Feb 2026,2723,100%,59%,42%,38%,36%,32%,29%,—,—,—,—,—
Mar 2026,2483,100%,54%,39%,34%,32%,27%,—,—,—,—,—,—
Apr 2026,2738,100%,55%,41%,35%,31%,—,—,—,—,—,—,—
May 2026,2026,100%,58%,40%,33%,—,—,—,—,—,—,—,—
